# Lab 11: Controllable Generation

**Goal:** Explore the latent space of our Conditional GAN and control image features.

What we'll do:
1. Interpolation between two images
2. Single-dimension exploration (what does each z dim control?)
3. Latent space arithmetic (thick - thin = thickness direction)
4. Conditional interpolation (morph 3 → 7)
5. Truncation trick (quality vs variety)

**Prerequisite:** We need a trained cGAN. If you ran Lab 10 in this folder, its saved generator (`cgan_generator.pt`) is loaded and training is skipped. Otherwise we train one first (same as Lab 10: ≈ 8 min on `mps`, ≈ 10 min on CPU with the 10k subset).

**How to use this lab:** answer each **Predict** prompt before running the cell.

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import os

torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available()
                      else "mps" if torch.backends.mps.is_available()  # Apple-silicon GPU
                      else "cpu")
print(f"Using: {device}")

## 0. Train a cGAN (Quick Recap)

Same architecture as Lab 10. We need a trained Generator for the experiments.

In [ ]:
# ── Data ──
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])
mnist = datasets.MNIST("data", train=True, download=True, transform=transform)
TRAIN_SUBSET = 10_000 if device.type == "cpu" else None   # only used if we have to train
if TRAIN_SUBSET:
    mnist = torch.utils.data.Subset(mnist, range(TRAIN_SUBSET))
loader = DataLoader(mnist, batch_size=64, shuffle=True)

# ── Models (same as Lab 10) ──
z_dim = 64
num_classes = 10
embed_dim = 50

class Generator(nn.Module):
    def __init__(self, z_dim=64, num_classes=10, embed_dim=50):
        super().__init__()
        self.label_embed = nn.Embedding(num_classes, embed_dim)
        self.model = nn.Sequential(
            nn.Linear(z_dim + embed_dim, 256 * 7 * 7),
            nn.Unflatten(1, (256, 7, 7)),
            nn.ConvTranspose2d(256, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 1, 4, 2, 1),
            nn.Tanh(),
        )

    def forward(self, z, labels):
        label_vec = self.label_embed(labels)
        x = torch.cat([z, label_vec], dim=1)
        return self.model(x)

class Discriminator(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.num_classes = num_classes
        self.model = nn.Sequential(
            nn.Conv2d(1 + num_classes, 64, 4, 2, 1),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 4, 2, 1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 1),
            nn.Sigmoid(),
        )

    def forward(self, images, labels):
        label_maps = self._make_label_maps(labels, images.size(2), images.size(3))
        x = torch.cat([images, label_maps], dim=1)
        return self.model(x)

    def _make_label_maps(self, labels, h, w):
        bs = labels.size(0)
        one_hot = torch.zeros(bs, self.num_classes, device=labels.device)
        one_hot.scatter_(1, labels.unsqueeze(1), 1.0)
        return one_hot.view(bs, self.num_classes, 1, 1).expand(-1, -1, h, w)

gen  = Generator(z_dim, num_classes, embed_dim).to(device)
disc = Discriminator(num_classes).to(device)
print("Models ready.")

In [ ]:
# ── Train ──
opt_G = torch.optim.Adam(gen.parameters(),  lr=2e-4, betas=(0.5, 0.999))
opt_D = torch.optim.Adam(disc.parameters(), lr=2e-4, betas=(0.5, 0.999))
criterion = nn.BCELoss()
epochs = 30

CKPT = "cgan_generator.pt"   # written by Lab 10
if os.path.exists(CKPT):
    gen.load_state_dict(torch.load(CKPT, map_location=device))
    print(f"Loaded trained generator from {CKPT} -- skipping training.")
else:
    print(f"Training cGAN for {epochs} epochs...")
    for epoch in range(epochs):
        for real_images, real_labels in loader:
            real_images = real_images.to(device)
            real_labels = real_labels.to(device)
            bs = real_images.size(0)
            real_t = torch.ones(bs, 1, device=device)
            fake_t = torch.zeros(bs, 1, device=device)

            # D step
            noise = torch.randn(bs, z_dim, device=device)
            fake = gen(noise, real_labels).detach()
            loss_D = criterion(disc(real_images, real_labels), real_t) + criterion(disc(fake, real_labels), fake_t)
            opt_D.zero_grad(); loss_D.backward(); opt_D.step()

            # G step
            noise = torch.randn(bs, z_dim, device=device)
            fake = gen(noise, real_labels)
            loss_G = criterion(disc(fake, real_labels), real_t)
            opt_G.zero_grad(); loss_G.backward(); opt_G.step()

        if epoch % 10 == 0 or epoch == epochs - 1:
            print(f"  Epoch {epoch:3d} | D: {loss_D:.4f} | G: {loss_G:.4f}")

    torch.save(gen.state_dict(), CKPT)

gen.eval()
print("\nGenerator ready (eval mode: BatchNorm uses its running statistics).")

---

## 1. Interpolation: Walking Between Two Images

Pick two noise vectors. Walk smoothly from one to the other.  
If the GAN is good, every step looks like a real digit.

**Predict:** will the 3s and the 9s built from the *same* noise pair share a style?

In [ ]:
def show_interpolation(gen, z1, z2, label, steps=10, title="Interpolation"):
    """Interpolate between z1 and z2, all with the same label."""
    fig, axes = plt.subplots(1, steps, figsize=(steps * 1.2, 1.5))
    labels = torch.full((1,), label, dtype=torch.long, device=device)

    with torch.no_grad():
        for i, alpha in enumerate(torch.linspace(0, 1, steps)):
            z = (1 - alpha) * z1 + alpha * z2
            img = gen(z, labels).cpu().squeeze()
            img = (img + 1) / 2
            axes[i].imshow(img, cmap="gray")
            axes[i].axis("off")
            if i == 0:
                axes[i].set_title("A", fontsize=10)
            elif i == steps - 1:
                axes[i].set_title("B", fontsize=10)

    plt.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()


# Interpolate between two 7s
z1 = torch.randn(1, z_dim, device=device)
z2 = torch.randn(1, z_dim, device=device)
show_interpolation(gen, z1, z2, label=7, title="Interpolation between two 7s")

# Try with different digits
z3 = torch.randn(1, z_dim, device=device)
z4 = torch.randn(1, z_dim, device=device)
show_interpolation(gen, z3, z4, label=3, title="Interpolation between two 3s")
show_interpolation(gen, z3, z4, label=9, title="Same noise journey, but as 9s")

## 2. Single-Dimension Exploration

Fix all z dimensions except ONE. Sweep that one from -3 to +3.  
See what that dimension controls.

**Predict:** out of 8 dimensions, how many will produce a visible, interpretable change?

In [ ]:
def explore_dimension(gen, z_base, dim, label, values=None):
    """Vary a single dimension of z and show the effect."""
    if values is None:
        values = [-3, -2, -1, 0, 1, 2, 3]

    labels = torch.full((1,), label, dtype=torch.long, device=device)
    fig, axes = plt.subplots(1, len(values), figsize=(len(values) * 1.2, 1.5))

    with torch.no_grad():
        for i, val in enumerate(values):
            z = z_base.clone()
            z[0, dim] = val
            img = gen(z, labels).cpu().squeeze()
            img = (img + 1) / 2
            axes[i].imshow(img, cmap="gray")
            axes[i].set_title(f"{val:+.0f}", fontsize=9)
            axes[i].axis("off")

    plt.suptitle(f"Dimension {dim} (label={label})", fontsize=11)
    plt.tight_layout()
    plt.show()


# Pick a base noise vector
z_base = torch.randn(1, z_dim, device=device)

# Explore first 8 dimensions for digit 7
print("Sweeping dimensions 0-7 for digit 7:")
print("Look for dimensions that control visible features!\n")
for d in range(8):
    explore_dimension(gen, z_base, dim=d, label=7)

In [ ]:
# Try the SAME dimensions but for a different digit
# If a dimension controls "thickness", it should work across digits
print("Same dimensions, but for digit 4:")
print("Do the same dimensions control the same features?\n")
for d in range(8):
    explore_dimension(gen, z_base, dim=d, label=4)

## 3. Latent Space Arithmetic

Find a "direction" in latent space by averaging.  

Strategy:
1. Generate many images with different z vectors
2. Measure a feature (we'll use pixel intensity as a proxy for thickness)
3. Average z for "high" group vs "low" group
4. Direction = high_avg - low_avg

**Predict:** a direction found only from 7s — will it also thicken 0s and 3s?

In [ ]:
# Generate 500 images of digit 7 and track their z vectors
n_samples = 500
label_7 = torch.full((1,), 7, dtype=torch.long, device=device)

all_z = []
all_intensities = []

with torch.no_grad():
    for _ in range(n_samples):
        z = torch.randn(1, z_dim, device=device)
        img = gen(z, label_7)

        # Measure "thickness" = average pixel intensity (brighter = more ink = thicker)
        intensity = img.mean().item()

        all_z.append(z)
        all_intensities.append(intensity)

all_z = torch.cat(all_z, dim=0)   # (500, 64)

# Sort by intensity
sorted_idx = sorted(range(n_samples), key=lambda i: all_intensities[i])

# Bottom 20% = "thin", Top 20% = "thick"
n_group = n_samples // 5
thin_z  = all_z[sorted_idx[:n_group]].mean(dim=0, keepdim=True)    # average of thin
thick_z = all_z[sorted_idx[-n_group:]].mean(dim=0, keepdim=True)   # average of thick

# The "thickness" direction
thickness_dir = thick_z - thin_z

print(f"Found thickness direction (norm: {thickness_dir.norm():.2f})")
print(f"Thin group avg intensity:  {sum(all_intensities[i] for i in sorted_idx[:n_group])/n_group:.3f}")
print(f"Thick group avg intensity: {sum(all_intensities[i] for i in sorted_idx[-n_group:])/n_group:.3f}")

In [ ]:
# Apply the thickness direction to different digits
print("Applying thickness direction to various digits:")
print("Left = thinner, Right = thicker\n")

scales = [-2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0]

for digit in [0, 3, 5, 7, 9]:
    labels = torch.full((1,), digit, dtype=torch.long, device=device)
    z_start = torch.randn(1, z_dim, device=device)

    fig, axes = plt.subplots(1, len(scales), figsize=(len(scales) * 1.2, 1.5))
    with torch.no_grad():
        for i, s in enumerate(scales):
            z = z_start + s * thickness_dir
            img = gen(z, labels).cpu().squeeze()
            img = (img + 1) / 2
            axes[i].imshow(img, cmap="gray")
            axes[i].set_title(f"{s:+.1f}", fontsize=9)
            axes[i].axis("off")

    plt.suptitle(f"Digit {digit}: thickness direction", fontsize=11)
    plt.tight_layout()
    plt.show()

## 4. Conditional Interpolation: Morph Between Digits

Keep the noise fixed. Blend the **label embeddings** to morph one digit into another.

**Predict:** will the midpoint between 3 and 8 look like a digit, or like neither?

In [ ]:
def conditional_interpolation(gen, z_fixed, label_a, label_b, steps=10):
    """Morph from label_a to label_b by blending embeddings."""
    fig, axes = plt.subplots(1, steps, figsize=(steps * 1.2, 1.5))

    with torch.no_grad():
        embed_a = gen.label_embed(torch.tensor([label_a], device=device))  # (1, 50)
        embed_b = gen.label_embed(torch.tensor([label_b], device=device))  # (1, 50)
        for i, alpha in enumerate(torch.linspace(0, 1, steps)):
            embed_mix = (1 - alpha) * embed_a + alpha * embed_b
            x = torch.cat([z_fixed, embed_mix], dim=1)
            img = gen.model(x).cpu().squeeze()
            img = (img + 1) / 2
            axes[i].imshow(img, cmap="gray")
            axes[i].axis("off")
            if i == 0:
                axes[i].set_title(str(label_a), fontsize=11, fontweight="bold")
            elif i == steps - 1:
                axes[i].set_title(str(label_b), fontsize=11, fontweight="bold")

    plt.suptitle(f"Morph: {label_a} → {label_b}", fontsize=12)
    plt.tight_layout()
    plt.show()


z_fixed = torch.randn(1, z_dim, device=device)

# Morph between different digit pairs
conditional_interpolation(gen, z_fixed, 3, 8)
conditional_interpolation(gen, z_fixed, 1, 7)
conditional_interpolation(gen, z_fixed, 0, 9)
conditional_interpolation(gen, z_fixed, 4, 6)

## 5. Truncation Trick: Quality vs Variety

Limit the range of z to trade diversity for quality.

As on the L11 slide we **clamp** z to [−t, t]. (StyleGAN-style truncation instead *re-samples* values outside the range, or pulls z toward the mean: `z = t * z`; clamping is the simplest version and piles a little probability onto ±t.)

**Predict:** at t = 0.3, how different will the 8 samples in a row be?

In [ ]:
truncation_values = [0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 3.0]
n_per = 8

fig, axes = plt.subplots(len(truncation_values), n_per,
                         figsize=(n_per * 1.2, len(truncation_values) * 1.4))

label_7 = torch.full((n_per,), 7, dtype=torch.long, device=device)

with torch.no_grad():
    for row, trunc in enumerate(truncation_values):
        # Sample noise and truncate
        z = torch.randn(n_per, z_dim, device=device)
        z = z.clamp(-trunc, trunc)

        fakes = gen(z, label_7).cpu()

        for col in range(n_per):
            img = (fakes[col].squeeze() + 1) / 2
            axes[row][col].imshow(img, cmap="gray")
            axes[row][col].axis("off")
        axes[row][0].text(-0.08, 0.5, f"t={trunc}", transform=axes[row][0].transAxes, ha='right', va='center', fontsize=10, fontweight='bold')

plt.suptitle("Truncation Trick: small t = high quality, low variety", fontsize=13)
plt.tight_layout()
plt.show()

print("t=0.3: very similar (low variety, high quality)")
print("t=3.0: very diverse (high variety, some weird ones)")

## 6. Bonus: Full Latent Space Map

Start at z = 0 (the centre of latent space), vary latent dimensions 0 and 1 from −2 to +2, and generate a 2D grid of images.  
This shows you a "slice" through the latent space.

In [ ]:
# Create a 7x7 grid by varying two dimensions
grid_size = 7
dim_x, dim_y = 0, 1    # which two dimensions to vary
label = 7
labels = torch.full((1,), label, dtype=torch.long, device=device)

z_center = torch.zeros(1, z_dim, device=device)  # start at origin
values = torch.linspace(-2, 2, grid_size)

fig, axes = plt.subplots(grid_size, grid_size, figsize=(8, 8))

with torch.no_grad():
    for row, vy in enumerate(values):
        for col, vx in enumerate(values):
            z = z_center.clone()
            z[0, dim_x] = vx
            z[0, dim_y] = vy
            img = gen(z, labels).cpu().squeeze()
            img = (img + 1) / 2
            axes[row][col].imshow(img, cmap="gray")
            axes[row][col].axis("off")

plt.suptitle(f"Latent Space Slice (dims {dim_x} vs {dim_y}, digit {label})", fontsize=13)
plt.tight_layout()
plt.show()

print(f"X-axis: dimension {dim_x} from -2 to +2")
print(f"Y-axis: dimension {dim_y} from -2 to +2")
print("Each image shows how these two dimensions affect the output.")

## Key Takeaways

1. **Interpolation** — smooth walks between z vectors = smooth image transitions
2. **Single-dim sweep** — some z dimensions control visible features (thickness, tilt)
3. **Latent arithmetic** — average z per group, subtract = feature direction
4. **Conditional interp** — blend label embeddings to morph between digits
5. **Truncation** — clamp z to trade variety for quality

**Day 2 complete!** Next: Day 3 (L12–L13) — how do we MEASURE if a GAN is good? (Inception Score, FID)